# Segmentation Benchmark

Evaluate segmentation models on HyperData validation datasets.

| Step | Description |
|------|-------------|
| §1 | Setup & connect to HyperData |
| §2 | Pull validation dataset |
| §3 | Build models (encoder+head, open-source segmenters) |
| §4 | Run benchmark |
| §5 | Visualize results |

## 1. Setup

In [ ]:
import os

# ---- Default HyperData remote configuration ----
os.environ.setdefault("HYPERDATA_ENDPOINT", "http://118.180.19.234:8021")
os.environ.setdefault("MINIO_ENDPOINT", "118.180.19.234")
os.environ.setdefault("MINIO_PORT", "9010")
os.environ.setdefault("MINIO_ACCESS_KEY", "hyperdata_admin")
os.environ.setdefault("MINIO_SECRET_KEY", "change-this-password-in-production")
os.environ.setdefault("S3_ENDPOINT", "118.180.19.234")
os.environ.setdefault("S3_PORT", "9010")
os.environ.setdefault("S3_ACCESS_KEY", "hyperdata_admin")
os.environ.setdefault("S3_SECRET_KEY", "change-this-password-in-production")
os.environ.setdefault("S3_BUCKET", "hyperdata-data")

import numpy as np
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 2. Pull Validation Dataset

Pull the `fibsem_val_dataset_sample` from HyperData S3.

The dataset contains 12 image-label pairs (1024x1024 grayscale TIFF),
stored as stacked Zarr arrays: `images (N,H,W)` + `masks (N,H,W)` + `dataset_meta`.

In [ ]:
from hyperdata import HyperData

REMOTE_URL = "s3://hyperdata-data/fibsem_val_dataset_sample"
LOCAL_DIR = "/tmp/fibsem_val_dataset_sample"

# Pull from remote S3
ds = HyperData(LOCAL_DIR)
ds.add_remote("origin", REMOTE_URL)
ds.pull("origin")
ds = HyperData(LOCAL_DIR)

print(f"Keys: {ds.keys()}")
print(f"Images: {ds['images'].shape}")
print(f"Masks:  {ds['masks'].shape}")

In [ ]:
# Load into benchmark dataset loader
from lumen.benchmark.dataset import ValDatasetLoader

val_ds = ValDatasetLoader(
    path=LOCAL_DIR,
    channels=1,
    image_size=512,  # resize for faster inference
)

print(f"Samples: {len(val_ds)}")
print(f"Names: {val_ds.names}")
print(f"Num classes: {val_ds.num_classes}")
print(f"Metadata: {val_ds.meta}")

In [ ]:
# Preview a sample
import matplotlib.pyplot as plt

sample = val_ds[0]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(sample.image[0].numpy(), cmap="gray")
ax1.set_title(f"{sample.name} — Image")
ax2.imshow(sample.mask.numpy(), cmap="tab10", interpolation="nearest")
ax2.set_title(f"{sample.name} — Mask")
plt.tight_layout()
plt.show()

## 3. Build Models

We benchmark multiple model types:

1. **Encoder + SegmentationHead** — local trained models or pretrained encoders
2. **Open-source segmenters** — SAM3, DINOv3, TIPSv2 via Lumen's registry

Adjust the cells below to add/remove models as needed.

In [ ]:
from lumen.benchmark.runner import BenchmarkRunner, ModelSpec
from lumen.models.encoder_base import EncoderBase
from lumen.models.heads import SegmentationHead

NUM_CLASSES = val_ds.num_classes
IMAGE_SIZE = 512

# --- Model 1: Tiny baseline encoder (random init) ---
class TinyEncoder(EncoderBase):
    """Minimal encoder for baseline comparison."""
    def __init__(self, patch_size=16, in_channels=1, embed_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.embed_dim = embed_dim
        self.supports_masked_tokens = False
        self.proj = torch.nn.Conv2d(
            in_channels, embed_dim, kernel_size=patch_size, stride=patch_size,
        )
        self.blocks = torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(
                d_model=embed_dim, nhead=4, dim_feedforward=256,
                batch_first=True, norm_first=True,
            ),
            num_layers=2,
        )
        self.norm = torch.nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.blocks(x)
        return self.norm(x)

tiny_enc = TinyEncoder(patch_size=16, in_channels=1, embed_dim=64)
tiny_head = SegmentationHead(
    embed_dim=64, num_classes=NUM_CLASSES, patch_size=16,
)

print(f"Tiny encoder params: {sum(p.numel() for p in tiny_enc.parameters()):,}")
print(f"Tiny head params: {sum(p.numel() for p in tiny_head.parameters()):,}")

In [ ]:
# --- Model 2 (optional): DINOv3 encoder from HuggingFace ---
# Uncomment to use a pretrained DINOv3 backbone.
# Requires: pip install transformers

# from lumen.models import build_encoder
# dinov3_enc = build_encoder(
#     "dinov3",
#     model_name="facebook/dinov2-small",
#     in_channels=1,
# )
# dinov3_head = SegmentationHead(
#     embed_dim=dinov3_enc.embed_dim,
#     num_classes=NUM_CLASSES,
#     patch_size=dinov3_enc.patch_size,
# )
# print(f"DINOv3 embed_dim={dinov3_enc.embed_dim}")

In [ ]:
# --- Model 3 (optional): SAM3 segmenter ---
# Uncomment to use SAM3 as a promptable segmenter.
# Requires: pip install transformers

# from lumen.models import build_segmenter
# sam3_seg = build_segmenter("sam3")
# print(f"SAM3 image_size={sam3_seg.image_size}")

## 4. Run Benchmark

In [ ]:
runner = BenchmarkRunner(val_ds, num_classes=NUM_CLASSES)

# Add the tiny baseline
runner.add_model(ModelSpec(
    name="tiny-baseline (random)",
    encoder=tiny_enc,
    head=tiny_head,
    device=DEVICE,
))

# Uncomment to add DINOv3:
# runner.add_model(ModelSpec(
#     name="dinov3-small",
#     encoder=dinov3_enc,
#     head=dinov3_head,
#     device=DEVICE,
# ))

# Uncomment to add SAM3:
# runner.add_model(ModelSpec(
#     name="sam3",
#     segmenter=sam3_seg,
#     text_prompts=["structure"],
#     device=DEVICE,
# ))

results = runner.run()

for r in results:
    print(f"\n{r.model_name}:")
    print(f"  mIoU:     {r.summary['mean_iou']:.4f}")
    print(f"  Dice:     {r.summary['mean_dice']:.4f}")
    print(f"  Pixel Acc: {r.summary['mean_pixel_acc']:.4f}")
    print(f"  Time:     {r.elapsed_seconds:.1f}s")

## 5. Visualize Results

In [ ]:
from lumen.benchmark.visualize import plot_summary_table, plot_predictions

# Summary table comparing all models
table_data = []
for r in results:
    entry = dict(r.summary)
    entry["model_name"] = r.model_name
    entry["elapsed_seconds"] = r.elapsed_seconds
    table_data.append(entry)

fig = plot_summary_table(table_data, title="FIB-SEM Segmentation Benchmark")
plt.show()

In [ ]:
# Per-sample predictions for the first model (show first 4 samples)
import matplotlib.pyplot as plt

first_result = results[0]
n_show = min(4, len(val_ds))

images = [val_ds[i].image for i in range(n_show)]
masks = [val_ds[i].mask for i in range(n_show)]
names = [val_ds[i].name for i in range(n_show)]

# Re-run predictions for visualization
preds = []
spec = runner._models[0]
for i in range(n_show):
    sample = val_ds[i]
    pred = runner._predict(spec, sample)
    preds.append(pred)

fig = plot_predictions(
    images, masks, preds, names,
    model_name=first_result.model_name,
)
plt.show()

In [ ]:
# Per-sample score breakdown
import pandas as pd

for r in results:
    print(f"\n=== {r.model_name} ===")
    df = pd.DataFrame(r.summary["per_sample"])
    print(df.to_string(index=False))
    print(f"\nMean — mIoU: {r.summary['mean_iou']:.4f}, "
          f"Dice: {r.summary['mean_dice']:.4f}, "
          f"PixAcc: {r.summary['mean_pixel_acc']:.4f}")